In [44]:
import sqlite3

#mss used for screenshotting 
from mss import mss

#Gym used Environment components
import gymnasium as gym
from gymnasium import Env
from gymnasium import spaces

print("all good")

#Imports
#Pytorch used for the neural nets
import torch
import torchvision

#NumPy used for operations (Transformational framework)
import numpy as np

#OpenCV used for frame processing
import cv2
import sys
print(sys.executable)
import torch
print("CUDA Available:", torch.cuda.is_available())
print("PyTorch CUDA Version:", torch.version.cuda)
#Matplotlib to visualize captured frames
from matplotlib import pyplot as plt

#time for pauses
import time
from collections import deque

%pip install pynput
#%pip install pillow==8.0.0
from pynput import mouse, keyboard
import time
%pip install websocket-client[optional]
import json
%pip install requests
import requests
import time

all good
c:\ProgramData\anaconda3\envs\new_env\python.exe
CUDA Available: True
PyTorch CUDA Version: 11.8
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import websocket
import json
import threading
import time
# === SETUP ===
latest_tosu_data = {}
data_lock = threading.Lock()

def start_tosu_websocket():
    def on_message(ws, message):
        with data_lock:
            global latest_tosu_data
            latest_tosu_data = json.loads(message)
    
    def on_error(ws, error):
        print(f"Tosu error: {error}")
    
    def run():
        while True:
            try:
                ws = websocket.WebSocketApp(
                    "ws://localhost:24050/websocket/v2",
                    on_message=on_message,
                    on_error=on_error
                )
                ws.run_forever()
            except Exception as e:
                print(f"WS crashed: {e}, reconnecting...")
                time.sleep(1)
    
    thread = threading.Thread(target=run, daemon=True)
    thread.start()

def get_tosu_data():
    with data_lock:
        return latest_tosu_data.copy() if latest_tosu_data else {}

# === USAGE ===
print("Starting Tosu WebSocket...")
start_tosu_websocket()
time.sleep(0.5)  # Wait for connection


Starting Tosu WebSocket...
Reading data at 10 FPS:
Frame 0: Combo=0, Acc=100.0%
Frame 1: Combo=0, Acc=100.0%
Frame 2: Combo=0, Acc=100.0%
Frame 3: Combo=0, Acc=100.0%
Frame 4: Combo=0, Acc=100.0%
Frame 5: Combo=0, Acc=100.0%
Frame 6: Combo=0, Acc=100.0%
Frame 7: Combo=0, Acc=100.0%
Frame 8: Combo=0, Acc=100.0%
Frame 9: Combo=0, Acc=100.0%
Frame 10: Combo=0, Acc=100.0%
Frame 11: Combo=0, Acc=100.0%
Frame 12: Combo=0, Acc=100.0%
Frame 13: Combo=0, Acc=100.0%
Frame 14: Combo=0, Acc=100.0%
Frame 15: Combo=0, Acc=100.0%
Frame 16: Combo=0, Acc=100.0%
Frame 17: Combo=0, Acc=100.0%
Frame 18: Combo=0, Acc=100.0%
Frame 19: Combo=0, Acc=100.0%
Frame 20: Combo=0, Acc=100.0%
Frame 21: Combo=0, Acc=100.0%
Frame 22: Combo=0, Acc=100.0%
Frame 23: Combo=0, Acc=100.0%
Frame 24: Combo=0, Acc=100.0%
Frame 25: Combo=0, Acc=100.0%
Frame 26: Combo=0, Acc=100.0%
Frame 27: Combo=0, Acc=100.0%
Frame 28: Combo=0, Acc=100.0%
Frame 29: Combo=0, Acc=100.0%
Frame 30: Combo=0, Acc=100.0%
Frame 31: Combo=0, Acc=100.0%

In [ ]:
# Quick test
while True:
    time.sleep(0.5)
    data = get_tosu_data()
    if data:
        print("✅ WebSocket connected!")
        #print(f"Keys available: {data.keys()}")
        print(data.values())
    else:
        print("❌ No data yet, wait a moment")

        

In [46]:
from pynput import mouse
from pynput.mouse import Button, Controller

In [ ]:
#run this at the beginning of each map

      
def parse_beatmap():

    '''parses all live beatmap into a list of tuples (list of hitobjects)
       index 0 = circle x positions, normalized
       index 1 = circle y positions, normalized
       index 2 = optimal time in milleseconds from map start to hit
       index 3 = approach rate raw
       index 4 = circle size radius in pixels raw
    '''
    hits = []
    in_hitobjects = False
    data = get_tosu_data() 
    approach_rate = data["beatmap"]["stats"]["ar"]["converted"] 
    full_osu_path = data["folders"]["songs"] + "\\" + data["directPath"]["beatmapFile"]

    with open(full_osu_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # wait until we reach the [HitObjects] section
            if line == '[HitObjects]':
                in_hitobjects = True
                continue

            # once in [HitObjects), parse each note
            
            if in_hitobjects and line:
                parts = line.split(',')
                #index 0 is x, index 2 is y, index 3 is time in milliseconds 

                # normalized to (0,1) vertically and horizontally 
                circle_x = int(2.1875*int(parts[0]))/(1120)
                circle_y = int(2.1875*int(parts[1]))/(720)
                time_start_ms = int(parts[2])
                time_end_ms = int()
                ar_ms = float()
                circle_size = data['beatmap']['stats']['cs']['converted']
                circle_size_pixels = int()
                
                #approach rate calculations to figure out the time to hit in parse_state_info
                if approach_rate is not None and approach_rate < 5:
                    ar_ms = int(1200 + 600*((5-approach_rate)/5)) 
                elif approach_rate is not None and approach_rate == 5:
                    ar_ms = int(1200)
                elif approach_rate is not None and approach_rate > 5:
                    ar_ms = int(1200 - 750 * ((approach_rate-5)/5))
                else:
                    print("an error occurred when calculating the AR")
                    break
                
                if circle_size is not None:
                    circle_size_pixels = (54.4 - 4.48 * circle_size)*2.8125
                else:
                    circle_size_pixels = (32*2.8125)

                hits.append((circle_x,circle_y,time_start_ms,ar_ms,circle_size_pixels))
                #sort by time so notes are in chronological order
        return hits


map_info = parse_beatmap()


#run this every 2nd frame

def parse_state_info(map_information):
    '''
    figure out the time to hit for the next 3 circles
    read the 300/100/50/miss/sliderbreak counts
    calculate the distance to the next 3 circles respectively
    find whether the cursor is inside of the radius of the hit circle for later reward
    find the combo and accuracy 
    put everything in one large dictionary

    '''
    data = get_tosu_data() 
    current_time_ms = data["beatmap"]["time"]["live"]
    upcoming = []

    #loop through all the hits 
    for hit in map_information:
        # h[2] is 'time_start_ms' 
        # keep notes that are starting now OR in the future.
        # The "- 50" gives a 50ms grace period so notes don't vanish instantly.
        if hit[2] >= (current_time_ms - 50):
            upcoming.append(hit)

    # only take the first 3 notes from that filtered list
    upcoming = upcoming[:3]

    if not upcoming:
        return {"state": "No notes found", "upcoming": []}

    circle_1_time_to_hit = upcoming[0][2] - current_time_ms 
    circle_2_time_to_hit = upcoming[1][2] - current_time_ms if len(upcoming) > 1 else 0
    circle_3_time_to_hit = upcoming[2][2] - current_time_ms if len(upcoming) > 2 else 0

    is_hovering_circle1 = False

    mouse = Controller()
    mx, my = mouse.position
    mx = mx - 160
    my = my




    if (((upcoming[0][0]*1120-mx)**2)+((upcoming[0][1]*720-my)**2))**0.5 < upcoming[0][4]:
        is_hovering_circle1 = True
    else:
        is_hovering_circle1 = False

    
    return {
        "current_time_ms": current_time_ms,
        "upcoming":        upcoming,
        "combo":           data["play"]["combo"]["current"],
        "accuracy":        data["play"]["accuracy"],
        "hp":              data["play"]["healthBar"]["normal"],
        "hits_300":        data["play"]["hits"]["300"],
        "hits_100":        data["play"]["hits"]["100"],
        "hits_50":         data["play"]["hits"]["50"],
        "misses":          data["play"]["hits"]["0"],
        "slider_breaks":   data["play"]["hits"]["sliderBreaks"],
        "state":           data["state"]["name"],
        'circle_1':        circle_1_time_to_hit,
        'circle_2':        circle_2_time_to_hit,
        'circle_3':        circle_3_time_to_hit,
        'is_hovering_circle1': is_hovering_circle1

    }


hits = parse_state_info(map_info)
                        

start_tosu_websocket()

# load the beatmap 
while True:
    hits = parse_state_info(map_info)
    print(f"Loaded {len(hits['upcoming'])} notes")
    for key, value in hits.items():
        print(f"Key: {key}, Value: {value}")
    time.sleep(5)

Loaded 0 notes
Key: state, Value: No notes found
Key: upcoming, Value: []
Loaded 3 notes
Key: current_time_ms, Value: -620
Key: upcoming, Value: [(0.01875, 0.5888888888888889, 43, 900, 102.6), (0.22589285714285715, 0.8527777777777777, 913, 900, 102.6), (0.71875, 0.8680555555555556, 1783, 900, 102.6)]
Key: combo, Value: 0
Key: accuracy, Value: 100
Key: hp, Value: 100
Key: hits_300, Value: 0
Key: hits_100, Value: 0
Key: hits_50, Value: 0
Key: misses, Value: 0
Key: slider_breaks, Value: 0
Key: state, Value: play
Key: circle_1, Value: 663
Key: circle_2, Value: 1533
Key: circle_3, Value: 2403
Key: is_hovering_circle1, Value: False
Loaded 3 notes
Key: current_time_ms, Value: 4100
Key: upcoming, Value: [(0.7026785714285714, 0.7472222222222222, 4609, 900, 102.6), (0.3455357142857143, 0.8652777777777778, 5261, 900, 102.6), (0.18482142857142858, 1.0569444444444445, 6782, 900, 102.6)]
Key: combo, Value: 12
Key: accuracy, Value: 100
Key: hp, Value: 90.46019879219655
Key: hits_300, Value: 4
Key: hi

KeyboardInterrupt: 